# Q5: Pattern Analysis

**Phase 6:** Pattern Analysis & Advanced Visualization  
**Points: 6 points**

**Focus:** Identify trends over time, analyze seasonal patterns, create
correlation analysis.

**Lecture Reference:** See **Lecture 11, Notebook 3**
(`11/demo/03_pattern_analysis_modeling_prep.ipynb`), Phase 6 for
examples of trend analysis, seasonal pattern identification, and
advanced visualizations. Also see **Lecture 09** for time series pattern
analysis.

In [17]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [18]:
print("="*60)
print("Q5: PATTERN ANALYSIS")
print("="*60)

# Load feature-engineered data from Q4 with datetime index
datetime_col = 'Measurement Timestamp'  # Update if different

df = pd.read_csv('output/q4_features.csv', 
                 parse_dates=[datetime_col], 
                 index_col=datetime_col)

print(f"\nLoaded {len(df):,} records with features")
print(f"Columns: {df.shape[1]}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

# Ensure data is sorted by datetime
df = df.sort_index()


Q5: PATTERN ANALYSIS

Loaded 195,892 records with features
Columns: 41
Date range: 2015-04-25 09:00:00 to 2025-11-24 12:00:00


In [19]:

# ========================================
# STEP 2: IDENTIFY NUMERIC COLUMNS FOR ANALYSIS
# ========================================
print("\n" + "="*60)
print("IDENTIFYING VARIABLES FOR ANALYSIS")
print("="*60)

# Get all numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nTotal numeric columns: {len(numeric_cols)}")

# Identify key sensor variables (exclude derived/temporal features for clarity)
# You may want to focus on original sensor readings
temporal_features = ['hour', 'day_of_week', 'month', 'year', 'is_weekend', 
                     'day_of_month', 'quarter']

# Key variables for analysis (adjust based on your data)
key_vars = [col for col in numeric_cols if col not in temporal_features 
            and not col.startswith('rolling') 
            and not col.endswith('_squared')
            and not col.endswith('_category')]

print(f"\nKey variables for pattern analysis: {len(key_vars)}")
for var in key_vars[:10]:  # Show first 10
    print(f"  - {var}")


IDENTIFYING VARIABLES FOR ANALYSIS

Total numeric columns: 33

Key variables for pattern analysis: 24
  - Air Temperature
  - Wet Bulb Temperature
  - Humidity
  - Rain Intensity
  - Interval Rain
  - Total Rain
  - Precipitation Type
  - Wind Direction
  - Wind Speed
  - Maximum Wind Speed


In [20]:
# ========================================
# STEP 3: TEMPORAL TREND ANALYSIS
# ========================================
print("\n" + "="*60)
print("STEP 1: TEMPORAL TREND ANALYSIS")
print("="*60)

print("\nAggregating data by time periods...")

# Monthly averages (use 'ME' for month end)
print("\nCalculating monthly averages...")
monthly_avg = df[key_vars].resample('ME').mean()
print(f"  Monthly data points: {len(monthly_avg)}")

# Daily averages
print("Calculating daily averages...")
daily_avg = df[key_vars].resample('D').mean()
print(f"  Daily data points: {len(daily_avg)}")

# Calculate overall trends
print("\nOverall trend analysis:")
for var in key_vars[:5]:  # Analyze first 5 variables
    overall_mean = df[var].mean()
    overall_std = df[var].std()
    overall_min = df[var].min()
    overall_max = df[var].max()
    print(f"\n{var}:")
    print(f"  Mean: {overall_mean:.2f}")
    print(f"  Std: {overall_std:.2f}")
    print(f"  Range: [{overall_min:.2f}, {overall_max:.2f}]")



STEP 1: TEMPORAL TREND ANALYSIS

Aggregating data by time periods...

Calculating monthly averages...
  Monthly data points: 128
Calculating daily averages...
  Daily data points: 3867

Overall trend analysis:

Air Temperature:
  Mean: 12.65
  Std: 10.43
  Range: [-29.78, 37.60]

Wet Bulb Temperature:
  Mean: 10.29
  Std: 9.40
  Range: [-28.90, 28.40]

Humidity:
  Mean: 68.02
  Std: 15.64
  Range: [0.00, 100.00]

Rain Intensity:
  Mean: 0.16
  Std: 1.80
  Range: [0.00, 183.60]

Interval Rain:
  Mean: 0.14
  Std: 1.10
  Range: [-0.90, 63.42]


In [21]:
# ========================================
# STEP 4: SEASONAL PATTERN ANALYSIS
# ========================================
print("\n" + "="*60)
print("STEP 2: SEASONAL PATTERN ANALYSIS")
print("="*60)

# Monthly patterns
print("\nAnalyzing monthly patterns...")
if 'month' in df.columns:
    monthly_pattern = df.groupby('month')[key_vars].mean()
    print("✓ Monthly patterns calculated")
    print("\nSample monthly pattern (first variable):")
    if len(key_vars) > 0:
        print(monthly_pattern[[key_vars[0]]].head())

# Hourly patterns (diurnal cycle)
print("\nAnalyzing hourly patterns (diurnal cycle)...")
if 'hour' in df.columns:
    hourly_pattern = df.groupby('hour')[key_vars].mean()
    print("✓ Hourly patterns calculated")
    print("\nSample hourly pattern (first variable):")
    if len(key_vars) > 0:
        print(hourly_pattern[[key_vars[0]]].head())

# Day of week patterns
print("\nAnalyzing day of week patterns...")
if 'day_of_week' in df.columns:
    dow_pattern = df.groupby('day_of_week')[key_vars].mean()
    print("✓ Day of week patterns calculated")


# Weekend vs weekday patterns
print("\nAnalyzing weekend vs weekday patterns...")
if 'is_weekend' in df.columns:
    weekend_pattern = df.groupby('is_weekend')[key_vars].mean()
    print("✓ Weekend patterns calculated")
    print("\nWeekend effect (first variable):")
    if len(key_vars) > 0:
        weekday_val = weekend_pattern.loc[0, key_vars[0]]
        weekend_val = weekend_pattern.loc[1, key_vars[0]]
        print(f"  Weekday: {weekday_val:.2f}")
        print(f"  Weekend: {weekend_val:.2f}")
        print(f"  Difference: {weekend_val - weekday_val:.2f}")
    


STEP 2: SEASONAL PATTERN ANALYSIS

Analyzing monthly patterns...
✓ Monthly patterns calculated

Sample monthly pattern (first variable):
       Air Temperature
month                 
1            -2.551031
2             0.501822
3             4.255540
4             8.633969
5            14.835429

Analyzing hourly patterns (diurnal cycle)...
✓ Hourly patterns calculated

Sample hourly pattern (first variable):
      Air Temperature
hour                 
0           12.508186
1           12.096588
2           11.810188
3           11.592512
4           11.382699

Analyzing day of week patterns...
✓ Day of week patterns calculated

Analyzing weekend vs weekday patterns...
✓ Weekend patterns calculated

Weekend effect (first variable):
  Weekday: 12.74
  Weekend: 12.43
  Difference: -0.31


In [22]:

# ========================================
# STEP 5: CORRELATION ANALYSIS
# ========================================
print("\n" + "="*60)
print("STEP 3: CORRELATION ANALYSIS")
print("="*60)

print("\nCalculating correlation matrix...")

# Select numeric columns for correlation (exclude categorical)
corr_cols = [col for col in numeric_cols if df[col].dtype in [np.float64, np.int64]]

# Calculate correlation matrix
corr_matrix = df[corr_cols].corr()
print(f"✓ Correlation matrix calculated ({corr_matrix.shape[0]}×{corr_matrix.shape[1]})")

# Find strongest correlations (excluding diagonal)
print("\nStrongest positive correlations (top 5):")
# Get upper triangle indices
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
corr_unstacked = corr_matrix.where(mask).stack().sort_values(ascending=False)
print(corr_unstacked.head(5))

print("\nStrongest negative correlations (top 5):")
print(corr_unstacked.tail(5))

# Store key correlations for summary
top_positive = corr_unstacked.head(3)
top_negative = corr_unstacked.tail(3)



STEP 3: CORRELATION ANALYSIS

Calculating correlation matrix...
✓ Correlation matrix calculated (33×33)

Strongest positive correlations (top 5):
Barometric Pressure  pressure_deviation      1.000000
Humidity             humidity_squared        0.989582
Air Temperature      Wet Bulb Temperature    0.980753
month                quarter                 0.968746
Wind Speed           wind_speed_squared      0.943087
dtype: float64

Strongest negative correlations (top 5):
Heading        Battery Life           -0.402526
comfort_index  humidity_rolling_24h   -0.556416
               humidity_rolling_7h    -0.659663
               humidity_squared       -0.701119
Humidity       comfort_index          -0.714054
dtype: float64


In [23]:

# ========================================
# SAVE ARTIFACT 1: q5_correlations.csv
# ========================================
print("\n" + "="*60)
print("SAVING ARTIFACTS")
print("="*60)

print("\nSaving correlation matrix...")
# Save full correlation matrix (or subset if too large)
if len(corr_cols) > 20:
    # Save top 20 most variable columns
    top_cols = df[corr_cols].std().nlargest(20).index.tolist()
    corr_subset = df[top_cols].corr()
    corr_subset.to_csv('output/q5_correlations.csv')
    print(f"✓ Saved: output/q5_correlations.csv (subset: {len(top_cols)} variables)")
else:
    corr_matrix.to_csv('output/q5_correlations.csv')
    print(f"✓ Saved: output/q5_correlations.csv ({corr_matrix.shape[0]} variables)")



SAVING ARTIFACTS

Saving correlation matrix...
✓ Saved: output/q5_correlations.csv (subset: 20 variables)


In [24]:

# ========================================
# CREATE ARTIFACT 2: q5_patterns.png
# ========================================
print("\nCreating pattern visualizations...")

# Create figure with 4 subplots
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Choose primary variable for visualization (first key variable)
primary_var = key_vars[0] if len(key_vars) > 0 else numeric_cols[0]
secondary_var = key_vars[1] if len(key_vars) > 1 else numeric_cols[1]

print(f"  Primary variable: {primary_var}")
print(f"  Secondary variable: {secondary_var}")

# Plot 1: Monthly trend over time (top-left, spans 2 columns)
ax1 = fig.add_subplot(gs[0, :])
if len(key_vars) >= 2:
    ax1.plot(monthly_avg.index, monthly_avg[primary_var], 
             linewidth=2, color='coral', marker='o', label=primary_var)
    ax1_twin = ax1.twinx()
    ax1_twin.plot(monthly_avg.index, monthly_avg[secondary_var], 
                  linewidth=2, color='steelblue', marker='s', label=secondary_var)
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel(primary_var, fontsize=12, color='coral')
    ax1_twin.set_ylabel(secondary_var, fontsize=12, color='steelblue')
    ax1.set_title('Temporal Trends: Monthly Averages', fontsize=14, fontweight='bold')
    ax1.grid(alpha=0.3)
    ax1.legend(loc='upper left')
    ax1_twin.legend(loc='upper right')
    ax1.tick_params(axis='y', labelcolor='coral')
    ax1_twin.tick_params(axis='y', labelcolor='steelblue')
else:
    ax1.plot(monthly_avg.index, monthly_avg[primary_var], 
             linewidth=2, color='coral', marker='o')
    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel(primary_var, fontsize=12)
    ax1.set_title('Temporal Trend: Monthly Average', fontsize=14, fontweight='bold')
    ax1.grid(alpha=0.3)

plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# Plot 2: Monthly seasonal pattern (middle-left)
ax2 = fig.add_subplot(gs[1, 0])
if 'month' in df.columns and len(key_vars) > 0:
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    ax2.bar(range(1, 13), monthly_pattern[primary_var], 
            color='skyblue', edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Month', fontsize=12)
    ax2.set_ylabel(primary_var, fontsize=12)
    ax2.set_title(f'Monthly Seasonal Pattern: {primary_var}', fontsize=12, fontweight='bold')
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels(month_names, rotation=45)
    ax2.grid(axis='y', alpha=0.3)

# Plot 3: Hourly diurnal pattern (middle-right)
ax3 = fig.add_subplot(gs[1, 1])
if 'hour' in df.columns and len(key_vars) > 0:
    ax3.plot(range(24), hourly_pattern[primary_var], 
             linewidth=2.5, color='darkorange', marker='o', markersize=6)
    ax3.set_xlabel('Hour of Day', fontsize=12)
    ax3.set_ylabel(primary_var, fontsize=12)
    ax3.set_title(f'Diurnal Pattern: {primary_var}', fontsize=12, fontweight='bold')
    ax3.set_xticks(range(0, 24, 3))
    ax3.grid(alpha=0.3)
    # Shade night hours
    ax3.axvspan(0, 6, alpha=0.1, color='gray', label='Night')
    ax3.axvspan(18, 24, alpha=0.1, color='gray')
    ax3.legend()

# Plot 4: Correlation heatmap (bottom, spans 2 columns)
ax4 = fig.add_subplot(gs[2, :])
# Select subset of variables for clearer heatmap
if len(corr_cols) > 10:
    # Show top 10 most correlated variables with primary variable
    if primary_var in corr_matrix.columns:
        top_corr_vars = corr_matrix[primary_var].abs().nlargest(11).index.tolist()
        heatmap_data = corr_matrix.loc[top_corr_vars, top_corr_vars]
    else:
        heatmap_data = corr_matrix.iloc[:10, :10]
else:
    heatmap_data = corr_matrix

sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, vmin=-1, vmax=1, square=True, 
            linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax4)
ax4.set_title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
plt.setp(ax4.get_yticklabels(), rotation=0)

# Overall title
fig.suptitle('Chicago Beach Weather Sensors - Pattern Analysis', 
             fontsize=16, fontweight='bold', y=0.995)

# Save figure
plt.savefig('output/q5_patterns.png', dpi=150, bbox_inches='tight')
print("✓ Saved: output/q5_patterns.png")
plt.close()



Creating pattern visualizations...
  Primary variable: Air Temperature
  Secondary variable: Wet Bulb Temperature
✓ Saved: output/q5_patterns.png


In [25]:
# ========================================
# SAVE ARTIFACT 3: q5_trend_summary.txt
# ========================================
print("\nCreating trend summary...")

# Calculate key statistics for summary
if 'month' in df.columns and len(key_vars) > 0:
    monthly_range = (monthly_pattern[primary_var].min(), 
                     monthly_pattern[primary_var].max())
    peak_month = monthly_pattern[primary_var].idxmax()
    low_month = monthly_pattern[primary_var].idxmin()
else:
    monthly_range = (df[primary_var].min(), df[primary_var].max())
    peak_month = "N/A"
    low_month = "N/A"

if 'hour' in df.columns and len(key_vars) > 0:
    peak_hour = hourly_pattern[primary_var].idxmax()
    low_hour = hourly_pattern[primary_var].idxmin()
else:
    peak_hour = "N/A"
    low_hour = "N/A"

# Write summary
with open('output/q5_trend_summary.txt', 'w') as f:
    f.write("KEY PATTERNS IDENTIFIED\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("TEMPORAL TRENDS:\n")
    f.write(f"- Primary variable analyzed: {primary_var}\n")
    f.write(f"- Overall mean: {df[primary_var].mean():.2f}\n")
    f.write(f"- Overall std: {df[primary_var].std():.2f}\n")
    f.write(f"- Overall range: [{df[primary_var].min():.2f}, {df[primary_var].max():.2f}]\n")
    f.write(f"- Monthly range: [{monthly_range[0]:.2f}, {monthly_range[1]:.2f}]\n")
    
    if peak_month != "N/A":
        month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April',
                      5: 'May', 6: 'June', 7: 'July', 8: 'August',
                      9: 'September', 10: 'October', 11: 'November', 12: 'December'}
        f.write(f"- Peak month: {month_names.get(peak_month, peak_month)}\n")
        f.write(f"- Lowest month: {month_names.get(low_month, low_month)}\n")
    
    f.write("\nDAILY PATTERNS:\n")
    if peak_hour != "N/A":
        f.write(f"- {primary_var} shows diurnal cycle\n")
        f.write(f"- Peak hour: {peak_hour}:00 ({hourly_pattern[primary_var].max():.2f})\n")
        f.write(f"- Minimum hour: {low_hour}:00 ({hourly_pattern[primary_var].min():.2f})\n")
        diurnal_range = hourly_pattern[primary_var].max() - hourly_pattern[primary_var].min()
        f.write(f"- Daily temperature range: {diurnal_range:.2f}\n")
    else:
        f.write("- Daily patterns not analyzed (hour feature not available)\n")
    
    f.write("\nCORRELATIONS:\n")
    f.write("Top positive correlations:\n")
    for idx, (pair, corr) in enumerate(top_positive.items(), 1):
        f.write(f"  {idx}. {pair[0]} vs {pair[1]}: {corr:.3f}\n")
    
    f.write("\nTop negative correlations:\n")
    for idx, (pair, corr) in enumerate(top_negative.items(), 1):
        f.write(f"  {idx}. {pair[0]} vs {pair[1]}: {corr:.3f}\n")
    
    f.write("\nKEY INSIGHTS:\n")
    if peak_month != "N/A" and peak_month in [6, 7, 8]:
        f.write(f"- {primary_var} peaks in summer months\n")
    if peak_month != "N/A" and peak_month in [12, 1, 2]:
        f.write(f"- {primary_var} lowest in winter months\n")
    if peak_hour != "N/A" and peak_hour in range(12, 17):
        f.write(f"- {primary_var} peaks in afternoon hours\n")
    if len(top_positive) > 0 and top_positive.iloc[0] > 0.7:
        f.write(f"- Strong positive correlation detected between key variables\n")

print("✓ Saved: output/q5_trend_summary.txt")



Creating trend summary...
✓ Saved: output/q5_trend_summary.txt


In [26]:
# ========================================
# VERIFICATION
# ========================================
print("\n" + "="*60)
print("VERIFICATION")
print("="*60)

# Verify files exist
print("\nVerifying output files:")
output_files = [
    'output/q5_correlations.csv',
    'output/q5_patterns.png',
    'output/q5_trend_summary.txt'
]
for file in output_files:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"  ✓ {file} ({size:.1f} KB)")
    else:
        print(f"  ✗ {file} NOT FOUND!")


VERIFICATION

Verifying output files:
  ✓ output/q5_correlations.csv (8.4 KB)
  ✓ output/q5_patterns.png (402.2 KB)
  ✓ output/q5_trend_summary.txt (0.9 KB)


In [27]:
# ========================================
# SUMMARY
# ========================================
print("\n" + "="*60)
print("Q5 COMPLETE - All artifacts created successfully!")
print("="*60)
print("\nFiles created:")
print("  1. output/q5_correlations.csv")
print("  2. output/q5_patterns.png")
print("  3. output/q5_trend_summary.txt")
print("\nPattern Analysis Summary:")
print(f"  - Variables analyzed: {len(key_vars)}")
print(f"  - Correlations calculated: {corr_matrix.shape[0]}×{corr_matrix.shape[1]}")
print(f"  - Temporal patterns: Monthly, Daily, Hourly")
print(f"  - Visualizations: 4 plots created")
print("\nKey Findings:")
print(f"  - Primary variable: {primary_var}")
if len(top_positive) > 0:
    print(f"  - Strongest correlation: {top_positive.iloc[0]:.3f}")
if peak_month != "N/A":
    print(f"  - Peak month: {peak_month}")
if peak_hour != "N/A":
    print(f"  - Peak hour: {peak_hour}:00")
print("\nNext: Proceed to Q6 for modeling preparation")
print("="*60)



Q5 COMPLETE - All artifacts created successfully!

Files created:
  1. output/q5_correlations.csv
  2. output/q5_patterns.png
  3. output/q5_trend_summary.txt

Pattern Analysis Summary:
  - Variables analyzed: 24
  - Correlations calculated: 33×33
  - Temporal patterns: Monthly, Daily, Hourly
  - Visualizations: 4 plots created

Key Findings:
  - Primary variable: Air Temperature
  - Strongest correlation: 1.000
  - Peak month: 7
  - Peak hour: 16:00

Next: Proceed to Q6 for modeling preparation
